In [1]:
# importazione librerie necessarie
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import gzip
import numpy as np

# modello da allenare (Decision Tree)
from sklearn.tree import DecisionTreeClassifier
# modello da allenare (Random Forest)
from sklearn.ensemble import RandomForestClassifier

# importazione metriche di valutazione e utilità al fine dell'allenamento dei modelli
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split

# per visualizzare il decision tree
from sklearn.tree import export_graphviz
from six import StringIO 
from IPython.display import Image  
import pydotplus

# per fare Oversampling/Undersampling
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

In [2]:
# features variabili su cui andare ad allenare i modelli (DA CAMBIARE PER RUN DIVERSE) [DATASET NON NORMALIZZATO]
features = ['avg_position','avg_position_1','avg_position_2','avg_position_3','avg_position_4','performance_wma', 'top20_consistency', 'races_dby_consistency','participants']
# features = ['avg_position','avg_position_1','avg_position_2','avg_position_3','performance_wma', 'top20_consistency', 'races_dby_consistency','participants']
# features = ['avg_position','avg_position_1','avg_position_2','performance_wma', 'top20_consistency', 'races_dby_consistency','participants']
# features = ['avg_position','avg_position_1','performance_wma', 'top20_consistency', 'races_dby_consistency','participants']
# features = ['avg_position','performance_wma', 'top20_consistency', 'races_dby_consistency','participants']

In [3]:
# features variabili su cui andare ad allenare i modelli (DA CAMBIARE PER RUN DIVERSE) [DATASET NORMALIZZATO]
features_n = ['avg_position_startlist','avg_position_1_startlist','avg_position_2_startlist','avg_position_3_startlist','avg_position_4_startlist','performance_wma', 'top20_consistency', 'races_dby_consistency','participants']
# features_n = ['avg_position_startlist','avg_position_1_startlist','avg_position_2_startlist','avg_position_3_startlist','performance_wma', 'top20_consistency', 'races_dby_consistency','participants']
# features_n = ['avg_position_startlist','avg_position_1_startlist','avg_position_2_startlist','performance_wma', 'top20_consistency', 'races_dby_consistency','participants']
# features_n = ['avg_position_startlist','avg_position_1_startlist','performance_wma', 'top20_consistency', 'races_dby_consistency','participants']
# features_n = ['avg_position_startlist','performance_wma', 'top20_consistency', 'races_dby_consistency','participants']

##  Not normalized dataset DT

In [4]:
# Take the merged dataset cleaned from null values and anomalies
with gzip.open('./dataset/new_datasets/train_set.pkl', 'r') as file:
    train_set = pickle.load(file)
with gzip.open('./dataset/new_datasets/test_set.pkl', 'r') as file:
    test_set = pickle.load(file)

FileNotFoundError: [Errno 2] No such file or directory: './dataset/new_datasets/train_set.pkl'

In [ ]:
train_set.info()

In [ ]:
test_set.info()

In [ ]:
print("Train_set shape before dropping NaN:",train_set.shape)
print("Test_set shape before dropping NaN:",test_set.shape)
train_set = train_set.dropna()
test_set = test_set.dropna()
print("Train_set shape post dropping NaN:",train_set.shape)
print("Test_set shape post dropping NaN:",test_set.shape)

X_train = train_set[features]
y_train = train_set['top20']

X_test = test_set[features]
y_test = test_set['top20']

print("X_train shape:",X_train.shape)
print("y_train shape:",y_train.shape)
print("X_test shape:",X_test.shape)
print("y_test shape:",y_test.shape)

### Allenamento con Grid Search

In [8]:
# modello da allenare
DT = DecisionTreeClassifier()

In [9]:
# parametri da testare
GRID_SEARCH_PARAMS = {     
    'criterion': ['gini', 'entropy'],   # Criterion per il calcolo dell'impurità
    'splitter': ['best', 'random'],     # Tipo di splitter da usare
    'max_depth': [None, 10, 20, 30],    # Limita la profondità dell'albero
    'min_samples_split': [2, 5, 10],    # Numero minimo di campioni per uno split
    'min_samples_leaf': [1, 2, 5],      # Numero minimo di campioni in una foglia
    'class_weight': [None, 'balanced'], # Pesi per le classi
    'random_state': [42]                # Fissare il seed per risultati riproducibili
}

In [10]:
# grid_search = GridSearchCV(estimator=DT, param_grid=GRID_SEARCH_PARAMS, cv=5, scoring='accuracy', verbose=1, n_jobs=-1)
grid_search = GridSearchCV(estimator=DT, param_grid=GRID_SEARCH_PARAMS, cv=5, scoring='f1', verbose=1, n_jobs=-1)

In [ ]:
grid_search.fit(X_train, y_train)

In [ ]:
print("Migliori Parametri:", grid_search.best_params_)
# questo valore rappresenta la media dell'F1-score sui fold (5 in questo caso) di validazione per la 
# combinazione di iperparametri che ha prodotto i risultati migliori.
print("Miglior Score:", grid_search.best_score_)

In [ ]:
best_params_DT = grid_search.best_params_
print("Migliori Parametri trovati:", best_params_DT)

In [14]:
best_DT = DecisionTreeClassifier(**best_params_DT)

In [ ]:
# Allena il modello sull'intero set di training
best_DT.fit(X_train, y_train)

## Normalized dataset DT

In [4]:
# Take the normalized merged dataset cleaned from null values and anomalies
with gzip.open('./dataset/new_datasets/train_set_normalized.pkl', 'r') as file:
    train_set_n = pickle.load(file)
with gzip.open('./dataset/new_datasets/test_set_normalized.pkl', 'r') as file:
    test_set_n = pickle.load(file)

In [ ]:
train_set_n.info()

In [ ]:
test_set_n.info()

In [ ]:
print("Train_set shape before dropping NaN:",train_set_n.shape)
print("Test_set shape before dropping NaN:",test_set_n.shape)
train_set_n = train_set_n.dropna()
test_set_n = test_set_n.dropna()
print("Train_set shape post dropping NaN:",train_set_n.shape)
print("Test_set shape post dropping NaN:",test_set_n.shape)

X_train = train_set_n[features_n]
y_train = train_set_n['top20']

X_test = test_set_n[features_n]
y_test = test_set_n['top20']

print("X_train shape:",X_train.shape)
print("y_train shape:",y_train.shape)
print("X_test shape:",X_test.shape)
print("y_test shape:",y_test.shape)

### Allenamento con Grid Search

In [8]:
# modello da allenare
DT2 = DecisionTreeClassifier()

In [9]:
# parametri da testare
GRID_SEARCH_PARAMS = {     
    'criterion': ['gini', 'entropy'],   # Criterion per il calcolo dell'impurità
    'splitter': ['best', 'random'],     # Tipo di splitter da usare
    'max_depth': [None, 10, 20, 30],    # Limita la profondità dell'albero
    'min_samples_split': [2, 5, 10],    # Numero minimo di campioni per uno split
    'min_samples_leaf': [1, 2, 5],      # Numero minimo di campioni in una foglia
    'class_weight': [None, 'balanced'], # Pesi per le classi
    'random_state': [42]                # Fissare il seed per risultati riproducibili
}

In [22]:
# grid_search = GridSearchCV(estimator=DT2, param_grid=GRID_SEARCH_PARAMS, cv=5, scoring='accuracy', verbose=1, n_jobs=-1)
grid_search_2 = GridSearchCV(estimator=DT2, param_grid=GRID_SEARCH_PARAMS, cv=5, scoring='f1', verbose=1, n_jobs=-1)

In [ ]:
grid_search_2.fit(X_train, y_train)

In [ ]:
print("Migliori Parametri:", grid_search_2.best_params_)
# questo valore rappresenta la media dell'F1-score sui fold (5 in questo caso) di validazione per la 
# combinazione di iperparametri che ha prodotto i risultati migliori.
print("Miglior Score:", grid_search_2.best_score_)

In [ ]:
best_params_DT2 = grid_search_2.best_params_
print("Migliori Parametri trovati:", best_params_DT2)

In [26]:
best_DT2 = DecisionTreeClassifier(**best_params_DT2)

In [ ]:
# Allena il modello sull'intero set di training
best_DT2.fit(X_train, y_train)

## Not Normalized dataset RF

In [28]:
# Take the merged dataset cleaned from null values and anomalies
with gzip.open('./dataset/new_datasets/train_set.pkl', 'r') as file:
    train_set = pickle.load(file)
with gzip.open('./dataset/new_datasets/test_set.pkl', 'r') as file:
    test_set = pickle.load(file)

In [ ]:
train_set.info()

In [ ]:
test_set.info()

In [ ]:
print("Train_set shape before dropping NaN:",train_set.shape)
print("Test_set shape before dropping NaN:",test_set.shape)
train_set = train_set.dropna()
test_set = test_set.dropna()
print("Train_set shape post dropping NaN:",train_set.shape)
print("Test_set shape post dropping NaN:",test_set.shape)

X_train = train_set[features]
y_train = train_set['top20']

X_test = test_set[features]
y_test = test_set['top20']

print("X_train shape:",X_train.shape)
print("y_train shape:",y_train.shape)
print("X_test shape:",X_test.shape)
print("y_test shape:",y_test.shape)

### Allenamento con Grid Search

In [32]:
# modello da allenare
RF = RandomForestClassifier()  

In [33]:
# parametri da testare
GRID_SEARCH_PARAMS = {     
    'n_estimators': [50, 100, 200],         # Numero di alberi nella foresta
    'criterion': ['gini', 'entropy'],       # Funzione per misurare la qualità dello split
    'max_depth': [None, 10, 20, 30],        # Profondità massima degli alberi
    'min_samples_split': [2, 5, 10],        # Numero minimo di campioni richiesti per dividere un nodo
    'min_samples_leaf': [1, 2, 5],          # Numero minimo di campioni per una foglia
    'class_weight': [None, 'balanced'],     # Pesi associati alle classi (utile per dati squilibrati)
    'random_state': [42]                    # Fissare il seed per risultati riproducibili
}

In [34]:
# grid_search_3 = GridSearchCV(estimator=RF, param_grid=GRID_SEARCH_PARAMS, cv=5, scoring='accuracy', verbose=1, n_jobs=-1)
grid_search_3 = GridSearchCV(estimator=RF, param_grid=GRID_SEARCH_PARAMS, cv=5, scoring='f1', verbose=1, n_jobs=-1)

In [ ]:
# 391 minuti!!
grid_search_3.fit(X_train, y_train)

In [ ]:
print("Migliori Parametri:", grid_search_3.best_params_)
# questo valore rappresenta la media dell'F1-score sui fold (5 in questo caso) di validazione per la 
# combinazione di iperparametri che ha prodotto i risultati migliori.
print("Miglior Score:", grid_search_3.best_score_)

In [ ]:
best_params_RF = grid_search_3.best_params_
print("Migliori Parametri trovati:", best_params_RF)

In [38]:
best_RF = RandomForestClassifier(**best_params_RF)

In [ ]:
# Allena il modello sull'intero set di training
best_RF.fit(X_train, y_train)

## Normalized dataset RF

In [13]:
# Take the normalized merged dataset cleaned from null values and anomalies
with gzip.open('./dataset/new_datasets/train_set_normalized.pkl', 'r') as file:
    train_set_n = pickle.load(file)
with gzip.open('./dataset/new_datasets/test_set_normalized.pkl', 'r') as file:
    test_set_n = pickle.load(file)

In [14]:
train_set_n.info()

<class 'pandas.core.frame.DataFrame'>
Index: 439070 entries, 0 to 439069
Data columns (total 37 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   _url_x                    439070 non-null  object        
 1   name_x                    439070 non-null  object        
 2   birth_year                439070 non-null  int64         
 3   weight                    439070 non-null  int64         
 4   height                    439070 non-null  int64         
 5   nationality               439070 non-null  object        
 6   race_id                   439070 non-null  object        
 7   name_y                    439070 non-null  object        
 8   points                    439070 non-null  int64         
 9   length                    439070 non-null  int64         
 10  profile                   439070 non-null  int64         
 11  startlist_quality         439070 non-null  int64         
 12  date   

In [15]:
test_set_n.info()

<class 'pandas.core.frame.DataFrame'>
Index: 35406 entries, 439070 to 474475
Data columns (total 37 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   _url_x                    35406 non-null  object        
 1   name_x                    35406 non-null  object        
 2   birth_year                35406 non-null  int64         
 3   weight                    35406 non-null  int64         
 4   height                    35406 non-null  int64         
 5   nationality               35406 non-null  object        
 6   race_id                   35406 non-null  object        
 7   name_y                    35406 non-null  object        
 8   points                    35406 non-null  int64         
 9   length                    35406 non-null  int64         
 10  profile                   35406 non-null  int64         
 11  startlist_quality         35406 non-null  int64         
 12  date             

In [16]:
print("Train_set shape before dropping NaN:",train_set_n.shape)
print("Test_set shape before dropping NaN:",test_set_n.shape)
train_set_n = train_set_n.dropna()
test_set_n = test_set_n.dropna()
print("Train_set shape post dropping NaN:",train_set_n.shape)
print("Test_set shape post dropping NaN:",test_set_n.shape)

X_train = train_set_n[features_n]
y_train = train_set_n['top20']

X_test = test_set_n[features_n]
y_test = test_set_n['top20']

print("X_train shape:",X_train.shape)
print("y_train shape:",y_train.shape)
print("X_test shape:",X_test.shape)
print("y_test shape:",y_test.shape)

Train_set shape before dropping NaN: (439070, 37)
Test_set shape before dropping NaN: (35406, 37)
Train_set shape post dropping NaN: (426364, 37)
Test_set shape post dropping NaN: (34509, 37)
X_train shape: (426364, 9)
y_train shape: (426364,)
X_test shape: (34509, 9)
y_test shape: (34509,)


### Allenamento con Grid Search

In [17]:
# modello da allenare
RF2 = RandomForestClassifier()

In [18]:
# parametri da testare
GRID_SEARCH_PARAMS = {     
    'n_estimators': [50, 100, 200],         # Numero di alberi nella foresta
    'criterion': ['gini', 'entropy'],       # Funzione per misurare la qualità dello split
    'max_depth': [None, 10, 20, 30],        # Profondità massima degli alberi
    'min_samples_split': [2, 5, 10],        # Numero minimo di campioni richiesti per dividere un nodo
    'min_samples_leaf': [1, 2, 5],          # Numero minimo di campioni per una foglia
    'class_weight': [None, 'balanced'],     # Pesi associati alle classi (utile per dati squilibrati)
    'random_state': [42]                    # Fissare il seed per risultati riproducibili
}

In [19]:
# grid_search = GridSearchCV(estimator=RF2, param_grid=GRID_SEARCH_PARAMS, cv=5, scoring='accuracy', verbose=1, n_jobs=-1)
grid_search_4 = GridSearchCV(estimator=RF2, param_grid=GRID_SEARCH_PARAMS, cv=5, scoring='f1', verbose=1, n_jobs=-1)

In [20]:
# 526 minuti!!!
grid_search_4.fit(X_train, y_train)

Fitting 5 folds for each of 432 candidates, totalling 2160 fits


GridSearchCV(cv=5, estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'class_weight': [None, 'balanced'],
                         'criterion': ['gini', 'entropy'],
                         'max_depth': [None, 10, 20, 30],
                         'min_samples_leaf': [1, 2, 5],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 200], 'random_state': [42]},
             scoring='f1', verbose=1)

In [22]:
best_params_RF2 = grid_search_4.best_params_
print("Migliori Parametri trovati:", best_params_RF2)

Migliori Parametri trovati: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200, 'random_state': 42}


In [24]:
best_RF2 = RandomForestClassifier(**best_params_RF2)

In [25]:
# Allena il modello sull'intero set di training
best_RF2.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10,
                       min_samples_leaf=2, n_estimators=200, random_state=42)